In [0]:
# ============================================================
# NOTEBOOK: nb_05_Gold
# PURPOSE:  Produces the two reporting outputs required by
#           the assessment and a reconciliation results table.
#
# OUTPUTS:
#   gold.report_completions_by_district
#   gold.report_enrolments_vs_completions
#   gold.reconciliation_results
#
# REPORTING PERIOD: Q4 2025
# CATALOG:          ktu_assessment_dev
# INPUT:            silver.participant_event_dedup
# ============================================================

import uuid
from datetime import datetime
from pyspark.sql import functions as F

CATALOG         = "ktu_assessment_dev"
AUDIT_SCHEMA    = "audit"
SILVER_SCHEMA   = "silver"
GOLD_SCHEMA     = "gold"

DEBUG = 1

run_id     = str(uuid.uuid4())
notebook   = "nb_05_Gold"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_05_GOLD STARTED")
    print("=" * 50)
    print(f"Run ID     : {run_id}")
    print(f"Start Time : {start_time}")

NB_05_GOLD STARTED
Run ID     : 24af7c1a-34fc-42e7-9298-519ee68e296b
Start Time : 2026-09-12 14:52:35.865173


In [0]:
# ============================================================
# AUDIT START
# ============================================================

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}', '{notebook}', 'gold', 'gold layer',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        NULL, 'RUNNING', 0, 0, 0, 'Gold in progress', 0
    )
""")

# ============================================================
# LOAD THE DEDUP FACT
# ============================================================
fact = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.participant_event_dedup")

if DEBUG:
    print(f"Fact rows: {fact.count()}")

Fact rows: 4832


In [0]:
# ============================================================
# BUILD REPORTING OUTPUTS
# ============================================================
#
# OUTPUT 1: completions by district
# OUTPUT 2: enrolments vs completions vs unique participants
#
# METRIC DEFINITIONS:
#   enrolment            = count of participant_event rows
#   completion           = count where is_completed = true
#   not_completed        = enrolment - completion
#   unique_participants  = distinct participant_key
#
# Both outputs include a TOTAL row.
# ============================================================

# ------------------------------------------------------------
# OUTPUT 1: COMPLETIONS BY DISTRICT
# ------------------------------------------------------------
completions_by_district = (
    fact.groupBy(F.coalesce(F.col("district_canonical"), F.lit("UNKNOWN")).alias("district"))
    .agg(
        F.count("*").alias("enrolments"),
        F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions"),
        F.countDistinct("participant_key").alias("unique_participants")
    )
    .withColumn("not_completed", F.col("enrolments") - F.col("completions"))
    .select("district", "enrolments", "completions", "not_completed", "unique_participants")
    .orderBy(F.desc("completions"))
)

# Compute totals separately, then union.
district_totals = fact.agg(
    F.lit("TOTAL").alias("district"),
    F.count("*").alias("enrolments"),
    F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions"),
    F.lit(None).cast("bigint").alias("not_completed"),
    F.countDistinct("participant_key").alias("unique_participants")
).withColumn(
    "not_completed", F.col("enrolments") - F.col("completions")
).select("district", "enrolments", "completions", "not_completed", "unique_participants")

report_by_district = completions_by_district.unionByName(district_totals)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.report_completions_by_district")
report_by_district.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.report_completions_by_district")

# ------------------------------------------------------------
# OUTPUT 2: ENROLMENTS VS COMPLETIONS BY SOURCE
# ------------------------------------------------------------
by_source = (
    fact.groupBy("source_system")
    .agg(
        F.count("*").alias("enrolments"),
        F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions"),
        F.countDistinct("participant_key").alias("unique_participants")
    )
    .withColumn("not_completed", F.col("enrolments") - F.col("completions"))
    .select("source_system", "enrolments", "completions", "not_completed", "unique_participants")
)

source_totals = fact.agg(
    F.lit("TOTAL").alias("source_system"),
    F.count("*").alias("enrolments"),
    F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions"),
    F.lit(None).cast("bigint").alias("not_completed"),
    F.countDistinct("participant_key").alias("unique_participants")
).withColumn(
    "not_completed", F.col("enrolments") - F.col("completions")
).select("source_system", "enrolments", "completions", "not_completed", "unique_participants")

report_by_source = by_source.unionByName(source_totals)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.report_enrolments_vs_completions")
report_by_source.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.report_enrolments_vs_completions")

if DEBUG:
    print("=" * 60)
    print("REPORT 1: COMPLETIONS BY DISTRICT")
    print("=" * 60)
    report_by_district.show(truncate=False)

    print()
    print("=" * 60)
    print("REPORT 2: ENROLMENTS VS COMPLETIONS BY SOURCE")
    print("=" * 60)
    report_by_source.show(truncate=False)

REPORT 1: COMPLETIONS BY DISTRICT
+--------------+----------+-----------+-------------+-------------------+
|district      |enrolments|completions|not_completed|unique_participants|
+--------------+----------+-----------+-------------+-------------------+
|Metro         |1738      |1630       |108          |1413               |
|UNKNOWN       |1199      |802        |397          |836                |
|Garden Route  |699       |532        |167          |455                |
|Cape Winelands|625       |465        |160          |396                |
|West Coast    |278       |196        |82           |144                |
|Overberg      |192       |152        |40           |141                |
|Central Karoo |101       |73         |28           |68                 |
|TOTAL         |4832      |3850       |982          |3259               |
+--------------+----------+-----------+-------------+-------------------+


REPORT 2: ENROLMENTS VS COMPLETIONS BY SOURCE
+--------------+----------+---

In [0]:
# ============================================================
# RECONCILIATION
# ============================================================
#
# CHECKS:
#   1. Source-level enrolments sum to district-level enrolments
#   2. Source-level completions sum to district-level completions
#   3. Source-level unique participants count equals district-level
#   4. Headline completions = district subtotal = source subtotal
#   5. Enrolments = completions + not_completed
# ============================================================

# Headline values from the fact table
headline = fact.agg(
    F.count("*").alias("enrolments"),
    F.sum(F.when(F.col("is_completed"), 1).otherwise(0)).alias("completions"),
    F.countDistinct("participant_key").alias("unique_participants")
).collect()[0]

headline_enrolments   = int(headline["enrolments"])
headline_completions  = int(headline["completions"])
headline_unique       = int(headline["unique_participants"])

# From district report (excluding TOTAL row)
district_sum = spark.sql(f"""
    SELECT
        SUM(enrolments) AS e,
        SUM(completions) AS c
    FROM {CATALOG}.{GOLD_SCHEMA}.report_completions_by_district
    WHERE district <> 'TOTAL'
""").collect()[0]

# From source report (excluding TOTAL row)
source_sum = spark.sql(f"""
    SELECT
        SUM(enrolments) AS e,
        SUM(completions) AS c
    FROM {CATALOG}.{GOLD_SCHEMA}.report_enrolments_vs_completions
    WHERE source_system <> 'TOTAL'
""").collect()[0]

# Build the reconciliation rows
checks = [
    ("headline_enrolments_source_sum",
     "Source subtotals sum to headline enrolments",
     headline_enrolments,
     int(source_sum["e"])),

    ("headline_enrolments_district_sum",
     "District subtotals sum to headline enrolments",
     headline_enrolments,
     int(district_sum["e"])),

    ("headline_completions_source_sum",
     "Source subtotals sum to headline completions",
     headline_completions,
     int(source_sum["c"])),

    ("headline_completions_district_sum",
     "District subtotals sum to headline completions",
     headline_completions,
     int(district_sum["c"])),

    ("enrolments_eq_completions_plus_not",
     "Enrolments = completions + not_completed",
     headline_enrolments,
     headline_enrolments),
]

rows = []
for check_name, description, expected, actual in checks:
    rows.append((
        run_id,
        check_name,
        description,
        str(expected),
        str(actual),
        "PASS" if expected == actual else "FAIL",
        None,
    ))

recon_df = spark.createDataFrame(
    rows,
    schema="run_id STRING, check_name STRING, description STRING, expected STRING, actual STRING, status STRING, message STRING"
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.reconciliation_results")
recon_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.reconciliation_results")

if DEBUG:
    print("=" * 60)
    print("RECONCILIATION RESULTS")
    print("=" * 60)
    recon_df.show(truncate=False)
    print(f"Headline enrolments       : {headline_enrolments}")
    print(f"Headline completions      : {headline_completions}")
    print(f"Headline unique           : {headline_unique}")
    print(f"District enrolments sum   : {int(district_sum['e'])}")
    print(f"District completions sum  : {int(district_sum['c'])}")
    print(f"Source enrolments sum     : {int(source_sum['e'])}")
    print(f"Source completions sum    : {int(source_sum['c'])}")

RECONCILIATION RESULTS
+------------------------------------+----------------------------------+----------------------------------------------+--------+------+------+-------+
|run_id                              |check_name                        |description                                   |expected|actual|status|message|
+------------------------------------+----------------------------------+----------------------------------------------+--------+------+------+-------+
|24af7c1a-34fc-42e7-9298-519ee68e296b|headline_enrolments_source_sum    |Source subtotals sum to headline enrolments   |4832    |4832  |PASS  |NULL   |
|24af7c1a-34fc-42e7-9298-519ee68e296b|headline_enrolments_district_sum  |District subtotals sum to headline enrolments |4832    |4832  |PASS  |NULL   |
|24af7c1a-34fc-42e7-9298-519ee68e296b|headline_completions_source_sum   |Source subtotals sum to headline completions  |3850    |3850  |PASS  |NULL   |
|24af7c1a-34fc-42e7-9298-519ee68e296b|headline_completions_distri

In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

fail_count = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {CATALOG}.{GOLD_SCHEMA}.reconciliation_results WHERE status = 'FAIL'
""").collect()[0]["n"]

overall_status = "SUCCESS" if fail_count == 0 else "FAILED"

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results WHERE run_id = '{run_id}'")

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
    VALUES (
        '{run_id}', 'gold_reconciliation', 'reconciliation',
        'gold.reconciliation_results',
        'All checks pass',
        '{fail_count} failed checks',
        '{ "PASS" if fail_count == 0 else "FAIL" }',
        NULL,
        current_timestamp()
    )
""")

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = '{overall_status}',
        rows_in          = {headline_enrolments},
        rows_out         = {headline_completions},
        rows_rejected    = 0,
        message          = 'Gold complete. Enrolments: {headline_enrolments}, Completions: {headline_completions}, Unique: {headline_unique}.',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print()
    print("=" * 50)
    print("GOLD SUMMARY")
    print("=" * 50)
    print(f"Run ID               : {run_id}")
    print(f"Enrolments           : {headline_enrolments}")
    print(f"Completions          : {headline_completions}")
    print(f"Not completed        : {headline_enrolments - headline_completions}")
    print(f"Unique participants  : {headline_unique}")
    print(f"Reconciliation fails : {fail_count}")
    print(f"Duration             : {duration}s")
    print(f"Status               : {overall_status}")
    print("=" * 50)

print("nb_05_Gold completed successfully.")


GOLD SUMMARY
Run ID               : 24af7c1a-34fc-42e7-9298-519ee68e296b
Enrolments           : 4832
Completions          : 3850
Not completed        : 982
Unique participants  : 3259
Reconciliation fails : 0
Duration             : 13s
Status               : SUCCESS
nb_05_Gold completed successfully.
